In [1]:
import numpy as np
import pandas as pd
import os
import glob
import re

# === CONFIGURATION ===
BASE_DIR = '..'
RAW_DIR = os.path.join(BASE_DIR, 'raw', 'brussels')
PROCESSED_DIR = os.path.join(BASE_DIR, 'processed', 'brussels')
REPORTS_DIR = os.path.join(BASE_DIR, 'reports')

os.makedirs(PROCESSED_DIR, exist_ok = True)
os.makedirs(REPORTS_DIR, exist_ok = True)

# Brussels Bounding Box
CITY_LAT_MIN, CITY_LAT_MAX = 50.78, 50.92
CITY_LON_MIN, CITY_LON_MAX = 4.25, 4.45

# Standard price cleaning function
def clean_price(price_value):
    if pd.isna(price_value): return np.nan      
    if isinstance(price_value, (int, float)): return float(price_value)
    s = str(price_value).strip()
    match = re.search(r"[-+]?[0-9,.]+", s.replace('$', ''))
    if not match: return np.nan
    num = match.group(0).replace(',', '') 
    try: return float(num)
    except: return np.nan

# === PROCESSING PIPELINE ===
snapshot_folders = [f for f in glob.glob(os.path.join(RAW_DIR, '*')) if os.path.isdir(f)]
qa_summary_list = []

print(f"--- START PROCESSING {len(snapshot_folders)} BRUSSELS SNAPSHOTS ---")

for folder_path in snapshot_folders:
    snapshot_name = os.path.basename(folder_path)
    print(f"\n>> Processing: {snapshot_name}")

    try:
        listings_df = pd.read_csv(os.path.join(folder_path, 'listings.csv.gz'), low_memory=False)
        calendar_df = pd.read_csv(os.path.join(folder_path, 'calendar.csv.gz'), low_memory=False)
        reviews_df = pd.read_csv(os.path.join(folder_path, 'reviews.csv.gz'), low_memory=False)
        neigh_df = pd.read_csv(os.path.join(folder_path, 'neighbourhoods.csv'))
    except FileNotFoundError:
        print(f"   [!] Missing file in folder {snapshot_name}, skipping.")
        continue

    # ---------------------------------------------------------
    # 1. LISTINGS PROCESSING (Basic & Coordinates)
    # ---------------------------------------------------------
    listings_df['price_numeric'] = listings_df['price'].apply(clean_price)
    
    # [QA1] Flag Price <= 0
    listings_df['qa_flag_price_zero'] = listings_df['price_numeric'].fillna(0) <= 0
    qa_summary_list.append({
        'snapshot_date': snapshot_name, 'rule_id': 'QA001_price_zero',
        'records_affected': int(listings_df['qa_flag_price_zero'].sum()),
        'handling_decision': 'Flagged'
    })

    # Datetime & Coordinates
    listings_df['host_since'] = pd.to_datetime(listings_df['host_since'], errors='coerce')
    listings_df['latitude'] = pd.to_numeric(listings_df['latitude'], errors='coerce')
    listings_df['longitude'] = pd.to_numeric(listings_df['longitude'], errors='coerce')

    # [QA2] Check Coordinates Out of Bounds
    listings_df['qa_flag_out_of_city'] = (
        (listings_df['latitude'] < CITY_LAT_MIN) | (listings_df['latitude'] > CITY_LAT_MAX) |
        (listings_df['longitude'] < CITY_LON_MIN) | (listings_df['longitude'] > CITY_LON_MAX)
    )
    qa_summary_list.append({
        'snapshot_date': snapshot_name, 'rule_id': 'QA002_coords_out_of_bounds',
        'records_affected': int(listings_df['qa_flag_out_of_city'].sum()),
        'handling_decision': 'Flagged'
    })

    # [QA3] Duplicate IDs
    dups = listings_df.duplicated(subset=['id']).sum()
    if dups > 0:
        listings_df = listings_df.drop_duplicates(subset=['id'], keep='first')
    qa_summary_list.append({
        'snapshot_date': snapshot_name, 'rule_id': 'QA003_duplicate_ids',
        'records_affected': int(dups),
        'handling_decision': 'Remove duplicates'
    })

    # ---------------------------------------------------------
    # 2. NEIGHBOURHOOD PROCESSING 
    # ---------------------------------------------------------
    if 'neighbourhood_group' in neigh_df.columns: 
        neigh_df.drop(columns=['neighbourhood_group'], inplace=True)
    
    # Get standard neighborhood list
    valid_neighbourhoods = set(neigh_df['neighbourhood'])
    
    # [QA4] Check if listing belongs to a valid neighborhood
    # (If the column in listings is 'neighbourhood_cleansed')
    col_neigh = 'neighbourhood_cleansed' if 'neighbourhood_cleansed' in listings_df.columns else 'neighbourhood'
    
    listings_df['qa_flag_invalid_neigh'] = ~listings_df[col_neigh].isin(valid_neighbourhoods)
    
    qa_summary_list.append({
        'snapshot_date': snapshot_name, 'rule_id': 'QA004_invalid_neighbourhood',
        'records_affected': int(listings_df['qa_flag_invalid_neigh'].sum()),
        'handling_decision': 'Flagged'
    })

    # ---------------------------------------------------------
    # 3. REVIEWS PROCESSING
    # ---------------------------------------------------------
    reviews_df['date'] = pd.to_datetime(reviews_df['date'], errors='coerce')
    
    # Remove reviews without content (NaN comments)
    initial_reviews = len(reviews_df)
    reviews_df = reviews_df.dropna(subset=['comments'])
    
    # [QA5] Filter out reviews of non-existent listings (Orphaned Reviews)
    # Only keep reviews whose listing_id exists in the filtered listings_df
    valid_ids = set(listings_df['id'])
    reviews_df = reviews_df[reviews_df['listing_id'].isin(valid_ids)]
    
    removed_reviews = initial_reviews - len(reviews_df)
    qa_summary_list.append({
        'snapshot_date': snapshot_name, 'rule_id': 'QA005_orphaned_or_empty_reviews',
        'records_affected': int(removed_reviews),
        'handling_decision': 'Removed'
    })

    # ---------------------------------------------------------
    # 4. CALENDAR PROCESSING
    # ---------------------------------------------------------
    calendar_df['date'] = pd.to_datetime(calendar_df['date'], errors='coerce')
    calendar_df['price_numeric'] = calendar_df['price'].apply(clean_price)
    if 'adjusted_price' in calendar_df.columns: calendar_df.drop(columns=['adjusted_price'], inplace=True)
    
    # Only keep calendar entries for valid listings (reduces file size)
    calendar_df = calendar_df[calendar_df['listing_id'].isin(valid_ids)]

    # ---------------------------------------------------------
    # 5. FILE EXPORT
    # ---------------------------------------------------------
    out_dir = os.path.join(PROCESSED_DIR, snapshot_name)
    os.makedirs(out_dir, exist_ok=True)
    
    listings_df.to_csv(os.path.join(out_dir, 'listings_processed.csv'), index=False)
    calendar_df.to_csv(os.path.join(out_dir, 'calendar_processed.csv'), index=False)
    reviews_df.to_csv(os.path.join(out_dir, 'reviews_processed.csv'), index=False)
    neigh_df.to_csv(os.path.join(out_dir, 'neighbourhoods_processed.csv'), index=False)
    
    print(f"    -> Successfully saved to: {out_dir}")

# Save QA report
pd.DataFrame(qa_summary_list).to_csv(os.path.join(REPORTS_DIR, 'qa_summary_brussels.csv'), index=False)
print("\n--- COMPLETED ---")

--- START PROCESSING 3 BRUSSELS SNAPSHOTS ---

>> Processing: 16 March, 2025
    -> Successfully saved to: ..\processed\brussels\16 March, 2025

>> Processing: 21 June, 2025
    -> Successfully saved to: ..\processed\brussels\21 June, 2025

>> Processing: 22 December, 2024
    -> Successfully saved to: ..\processed\brussels\22 December, 2024

--- COMPLETED ---
